In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:

# Task 1: Write your code here:
df.isnull().sum()

nums = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
nums
cat= df.select_dtypes(include=["object"]).columns
for col in nums:
      df[col]=df[col].fillna(df[col].mean())
for col in cat:

    df[col]=df[col].fillna(df[col].mode()[0])

df.isnull().sum()

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Duplicates: {duplicates}")
if(duplicates>0):
    df.drop_duplicates(inplace= True)
    duplicates = df.duplicated().sum()
    print(f"After removing Duplicates: {duplicates}")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in cat:
   df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import  StandardScaler

scaler = StandardScaler()
df2=df.copy()
df2[nums] = scaler.fit_transform(df[nums])

In [ ]:
# Task 5: Write your code here:
df2['Target'].value_counts()
#There is an imbalance

In [ ]:
# Task 1: Write your code here:
X = df2.drop('Target', axis=1).astype(float)
y = df2["Target"].astype(float)
df2.isnull().sum()

In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from catboost import CatBoostClassifier

folds = []
fold_f1 = []

catWithBoots=CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  catWithBoots.fit(X_train, y_train) # train
  y_pred = catWithBoots.predict(X_test) # validate

  accuracy = accuracy_score(y_test, y_pred)
  fofo = f1_score(y_test, y_pred)
  fold_f1.append(fofo)
  folds.append(accuracy)
  print(folds)

  print(f"eval : {fold_idx + 1} \n accuracy: ",accuracy,' F1 score: ',fofo)

In [ ]:
print(f"accuracy avg:  {np.average(folds)}  F1 score avg: {np.average(fold_f1)}")

In [ ]:
# Task 1: Write your code here:
importances = catWithBoots.feature_importances_
importances=importances[importances>3]
absolute_coef = np.abs(importances)
sorted_idx = np.argsort(absolute_coef)

plt.figure(figsize=(15, 6))
features = X.columns
plt.bar(features[sorted_idx], importances[sorted_idx],color='pink',edgecolor='purple')
plt.xlabel("Coefficient Value (Impact)")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
print('The golden feature issss: -Drums please-\n','P_2')

In [ ]:
# Task Bonus: Write your code here:
